# 00_Utils — Ontologie métier ENTRADE.MA

**Objectif** : définir une fois pour toutes l'ontologie métier (institutions,
populations cibles, régions valides, IDs stables, extraction d'institutions
depuis du texte libre) et l'exporter comme module Python (`etl_lib/ontology.py`)
importé par tous les notebooks suivants (`01` à `06`).

**Pourquoi un module et pas seulement des cellules ?** Dans la version
précédente, chaque notebook réimplémentait sa propre logique de matching
avec des hypothèses différentes sur la structure des données sources —
plusieurs bugs silencieux en ont résulté (voir `01_Normalize.ipynb` pour le
détail). Centraliser cette logique dans un module testable et réutilisé
partout élimine cette classe de bugs.

## 1. Chemins de travail

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
RAG_DATA_DIR = NOTEBOOK_DIR.parent / "rag data"
DATA_DIR = NOTEBOOK_DIR.parent / "data"
PROCESSED_DIR = DATA_DIR / "processed"
CHUNKS_DIR = DATA_DIR / "chunks"
EMBEDDINGS_DIR = DATA_DIR / "embeddings"
RELATIONS_DIR = DATA_DIR / "relations"
REPORTS_DIR = DATA_DIR / "reports"

for d in (PROCESSED_DIR, CHUNKS_DIR, EMBEDDINGS_DIR, RELATIONS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(NOTEBOOK_DIR))

assert RAG_DATA_DIR.exists(), f"Introuvable : {RAG_DATA_DIR}"
print(f"rag data/  -> {RAG_DATA_DIR}  ({len(list(RAG_DATA_DIR.glob('*')))} entrées)")
print(f"data/      -> {DATA_DIR}")


rag data/  -> /home/ubunto/l_backup (3)/rag data  (13 entrées)
data/      -> /home/ubunto/l_backup (3)/data


## 2. Ontologie métier

Définie dans `etl_lib/ontology.py` (module versionné, pas seulement dans ce
notebook) :
- **`INSTITUTION_MAP`** : les 11 types d'établissements (CEF, CFA, CAPE, UPE,
  EPS, CJPA, COAPH, CPSH, CRECHE, JE, CAREPS), avec code de polyvalence
  (1 = spécialisé, 5 = EPS = dernier recours universel).
- **`POPULATION_CIBLES`** : les 6 populations, avec ordre de priorité métier
  (enfants > femmes > personnes âgées > personnes handicapées > jeunes > famille).
- **`MOROCCAN_REGIONS`** : les 12 régions officielles (validation géographique
  de `centres.jsonl`, source bruitée).
- **`stable_id()`** : hash déterministe (même entrée -> même ID partout,
  condition nécessaire à la cohérence Neo4j <-> Qdrant).
- **`extract_institutions()`** : reconnaît les codes institution dans du texte
  libre (ex: `"مراكز المواكبة لحماية الطفولة (CAPE)"` -> `["CAPE"]`), avec
  repli EPS explicitement flagué (`eps_fallback=True`) si rien n'est détecté.

In [2]:
from etl_lib.ontology import (
    INSTITUTION_MAP, POPULATION_CIBLES, MOROCCAN_REGIONS, EPS_CODE,
    clean_text, normalize_arabic, stable_id, extract_institutions, match_population,
)

print(f"Institutions   : {len(INSTITUTION_MAP)} codes -> {list(INSTITUTION_MAP)}")
print(f"Populations    : {len(POPULATION_CIBLES)} codes -> {list(POPULATION_CIBLES)}")
print(f"Régions valides: {len(MOROCCAN_REGIONS)}")


Institutions   : 11 codes -> ['CEF', 'CFA', 'CAPE', 'UPE', 'EPS', 'CJPA', 'COAPH', 'CPSH', 'CRECHE', 'JE', 'CAREPS']
Populations    : 6 codes -> ['enfants', 'femmes', 'personnes_agees', 'personnes_handicapees', 'jeunes', 'famille']
Régions valides: 12


## 3. Auto-test sur des exemples réels

Vérification rapide que l'extraction fonctionne sur des formulations
effectivement rencontrées dans `rag data/` (et pas seulement sur des
exemples inventés).

In [3]:
test_cases = [
    ("مراكز المواكبة لحماية الطفولة (CAPE)\nوحدات حماية الطفولة (UPE/ASS)", ["CAPE", "UPE"]),
    ("CEF", ["CEF"]),
    ("حضانة اجتماعية", ["CRECHE"]),
    ("", ["EPS"]),  # aucune institution détectée -> repli EPS explicite
]

all_ok = True
for text, expected in test_cases:
    codes, eps_fallback = extract_institutions(text)
    ok = codes == expected
    all_ok &= ok
    status = "OK" if ok else "ECHEC"
    print(f"[{status}] {text[:40]!r:45} -> {codes} (eps_fallback={eps_fallback})")

assert all_ok, "L'extraction d'institutions ne matche pas les cas attendus"
print("\nOntologie prête. Modules exportés dans etl_lib/ — importables depuis 01_Normalize.ipynb et suivants.")


[OK] 'مراكز المواكبة لحماية الطفولة (CAPE)\nوحد'   -> ['CAPE', 'UPE'] (eps_fallback=False)
[OK] 'CEF'                                         -> ['CEF'] (eps_fallback=False)
[OK] 'حضانة اجتماعية'                              -> ['CRECHE'] (eps_fallback=False)
[OK] ''                                            -> ['EPS'] (eps_fallback=True)

Ontologie prête. Modules exportés dans etl_lib/ — importables depuis 01_Normalize.ipynb et suivants.
